# 🌍 Climate & Health - Geospatial Optimized Model
## Enhanced with Location Intelligence + Hyperparameter Optimization

**Goal:** Maximize **Final Score = 0.60 × F1 + 0.40 × ROC-AUC**

### Key Optimizations:
1. **Advanced Geospatial Features** - Clustering, distance encoding, location risk
2. **Target Encoding** - Location-based risk scores
3. **SMOTE** - Synthetic oversampling for class imbalance
4. **Hyperparameter Optimization** - Grid search for key models
5. **Improved Stacking** - Better meta-learner and model selection
6. **Location Clustering** - KMeans on coordinates for spatial patterns

## 📚 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_predict, GridSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, 
    StackingClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, roc_auc_score, classification_report

# Advanced models
try:
    import xgboost as xgb
    HAS_XGB = True
    print("✓ XGBoost loaded")
except ImportError:
    HAS_XGB = False
    print("✗ XGBoost not available")

try:
    import lightgbm as lgb
    HAS_LGBM = True
    print("✓ LightGBM loaded")
except ImportError:
    HAS_LGBM = False
    print("✗ LightGBM not available")

try:
    import catboost as cb
    HAS_CAT = True
    print("✓ CatBoost loaded")
except ImportError:
    HAS_CAT = False
    print("✗ CatBoost not available")

# SMOTE for imbalanced data
try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
    print("✓ SMOTE available")
except ImportError:
    HAS_SMOTE = False
    print("✗ SMOTE not available - install with: pip install imbalanced-learn")

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize_scalar
import geopy.distance

sns.set_theme(style="whitegrid", font_scale=1.1)
pd.set_option("display.max_columns", 150)
np.random.seed(42)

## ⚙️ Configuration

In [ ]:
RANDOM_STATE = 42
N_SPLITS = 5
TARGET = "is_climate_sensitive"
ID_COL = "ID"
N_CLUSTERS = 8  # For geospatial clustering

## 📥 Load and Merge Data

In [ ]:
print("=" * 60)
print("LOADING DATA")
print("=" * 60)

train = pd.read_csv("Train.csv")
test = pd.read_csv("Test.csv")
climate_features = pd.read_csv("climate_features.csv")
climate_features = climate_features.drop(columns=["deathdate"], errors='ignore')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

# Merge
train = train.merge(climate_features, on=ID_COL, how="left")
test = test.merge(climate_features, on=ID_COL, how="left")

print(f"Merged Train: {train.shape}")
print(f"Merged Test: {test.shape}")

# Target distribution
print(f"\nTarget Distribution:")
target_dist = train[TARGET].value_counts(normalize=True).round(4)
print(target_dist)
print(f"Positive class: {100*target_dist.iloc[0]:.1f}%")

## 🗺️ ADVANCED GEOSPATIAL FEATURE ENGINEERING

In [ ]:
def create_geospatial_features(df, train_stats=None, is_train=True):
    """Create advanced geospatial features"""
    df = df.copy()
    
    # Basic coordinates
    lat = df['latitude'].values
    lon = df['longitude'].values
    
    # Distance from center of Uganda (approximate)
    UGANDA_CENTER = (1.3733, 32.2903)
    df['dist_from_center'] = np.sqrt((lat - UGANDA_CENTER[0])**2 + (lon - UGANDA_CENTER[1])**2) * 111  # km approx
    
    # Distance from equator
    df['dist_from_equator'] = np.abs(lat) * 111  # km
    
    # Coordinate interactions
    df['lat_lon_interaction'] = lat * lon
    df['lat_squared'] = lat ** 2
    df['lon_squared'] = lon ** 2
    
    # Regional encoding
    df['north_south'] = (lat > 1.0).astype(int)  # Northern vs Southern
    df['east_west'] = (lon > 33.5).astype(int)  # Eastern vs Western
    df['region_quad'] = df['north_south'] * 2 + df['east_west']  # 4 quadrants
    
    # Fine-grained binning
    df['lat_bin_fine'] = pd.cut(lat, bins=10, labels=False, duplicates='drop')
    df['lon_bin_fine'] = pd.cut(lon, bins=10, labels=False, duplicates='drop')
    df['location_hash'] = df['lat_bin_fine'].astype(str) + '_' + df['lon_bin_fine'].astype(str)
    
    return df


def deep_feature_engineering(df, train_target_stats=None, is_train=True):
    """Comprehensive feature engineering with geospatial focus"""
    df = df.copy()
    
    # Convert date
    df["deathdate"] = pd.to_datetime(df["deathdate"], errors="coerce")
    
    # ========== GEOSPATIAL FEATURES ==========
    df = create_geospatial_features(df, train_target_stats, is_train)
    
    # ========== TEMPORAL FEATURES ==========
    df["day_of_year"] = df["deathdate"].dt.dayofyear
    df["month"] = df["deathdate"].dt.month
    df["season"] = ((df["month"] % 12 + 3) // 3) % 4
    
    # Cyclical encoding
    df["day_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365.25)
    df["day_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365.25)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    
    # ========== AGE FEATURES ==========
    df["age_group"] = np.digitize(df["age"], bins=[1, 5, 12, 18, 30, 45, 60, 75])
    df["is_vulnerable_age"] = ((df["age"] < 5) | (df["age"] > 65)).astype(int)
    df["is_elderly"] = (df["age"] >= 60).astype(int)
    df["is_child"] = (df["age"] < 12).astype(int)
    df["is_infant"] = (df["age"] < 1).astype(int)
    df["age_log"] = np.log1p(df["age"])
    
    # ========== TEMPERATURE FEATURES ==========
    df["temperature_range"] = df["max_temperature"] - df["min_temperature"]
    df["temp_extreme_high"] = (df["avg_temperature"] > 25).astype(int)
    df["temp_extreme_low"] = (df["avg_temperature"] < 18).astype(int)
    df["temp_anomaly"] = df["avg_temperature"] - df["tavg_30d"]
    df["temp_anomaly_abs"] = df["temp_anomaly"].abs()
    df["temp_variability"] = df["temp_range_mean_30d"] / (df["avg_temperature"] + 1)
    df["max_temp_extreme"] = (df["max_temperature"] > 30).astype(int)
    df["min_temp_extreme"] = (df["min_temperature"] < 15).astype(int)
    
    # Temperature trends
    df["temp_trend_7d"] = df["tavg_7d"] - df["tavg_30d"]
    df["temp_warming"] = (df["temp_trend_7d"] > 1).astype(int)
    df["temp_cooling"] = (df["temp_trend_7d"] < -1).astype(int)
    
    # ========== PRECIPITATION FEATURES ==========
    df["is_heavy_rain"] = (df["precipitation"] > 5).astype(int)
    df["is_rainy_day"] = (df["precipitation"] > 0.1).astype(int)
    df["rain_intensity"] = df["precipitation"] / (df["rain_days_30d"] + 1)
    df["rain_anomaly"] = df["rain_sum_7d"] - (df["rain_sum_30d"] / 30 * 7)
    df["rain_anomaly_abs"] = df["rain_anomaly"].abs()
    df["recent_rain_ratio"] = df["rain_sum_7d"] / (df["rain_sum_30d"] + 1)
    df["rain_frequency"] = df["rain_days_30d"] / 30
    df["prolonged_rain"] = (df["rain_days_30d"] > 20).astype(int)
    df["drought_period"] = (df["rain_days_30d"] < 5).astype(int)
    df["heavy_rain_event"] = (df["max_daily_rain_30d"] > 20).astype(int)
    
    # ========== NDVI FEATURES ==========
    df["ndvi_change"] = df["ndvi_30d"] - df["ndvi_90d"]
    df["ndvi_high"] = (df["ndvi_30d"] > 0.7).astype(int)
    df["ndvi_low"] = (df["ndvi_30d"] < 0.4).astype(int)
    df["ndvi_declining"] = (df["ndvi_change"] < -0.1).astype(int)
    df["vegetation_stress"] = ((df["ndvi_30d"] < 0.5) & (df["avg_temperature"] > 23)).astype(int)
    
    # ========== ELEVATION/TERRAIN ==========
    df["high_elevation"] = (df["elevation"] > 1500).astype(int)
    df["elevation_log"] = np.log1p(df["elevation"])
    df["steep_slope"] = (df["slope"] > 5).astype(int)
    df["slope_log"] = np.log1p(df["slope"])
    df["elevation_temp"] = df["elevation"] * df["avg_temperature"] / 1000
    
    # ========== HEAT DAYS ==========
    df["heat_wave"] = (df["hot_days_30d"] > 5).astype(int)
    df["extreme_heat"] = (df["hot_days_30d"] > 10).astype(int)
    df["heat_frequency"] = np.log1p(df["hot_days_30d"])
    
    # ========== INTERACTION FEATURES ==========
    df["elderly_heat"] = df["is_elderly"] * df["temp_extreme_high"]
    df["child_rain"] = df["is_child"] * df["is_rainy_day"]
    df["vulnerable_extreme"] = df["is_vulnerable_age"] * df["temp_extreme_high"]
    df["hot_humid"] = df["temp_extreme_high"] * df["is_rainy_day"]
    df["cold_wet"] = df["temp_extreme_low"] * df["is_heavy_rain"]
    df["hot_dry"] = df["temp_extreme_high"] * df["drought_period"]
    df["high_elev_cold"] = df["high_elevation"] * df["temp_extreme_low"]
    df["slope_rain"] = df["steep_slope"] * df["is_heavy_rain"]
    df["vegetation_heat"] = df["ndvi_high"] * df["temp_extreme_high"]
    df["triple_risk"] = df["is_elderly"] * df["heat_wave"] * df["drought_period"]
    
    # ========== COMPOSITE SCORES ==========
    df["climate_stress_score"] = (
        df["hot_days_30d"] / 10 +
        df["temp_anomaly_abs"] * 2 +
        df["rain_anomaly_abs"] / 10 +
        df["is_vulnerable_age"] * 2 +
        df["heat_wave"] * 1.5
    )
    
    df["environmental_risk"] = (
        df["temp_extreme_high"].astype(float) +
        df["is_heavy_rain"].astype(float) +
        df["ndvi_low"].astype(float) +
        df["high_elevation"].astype(float)
    )
    
    # ========== GENDER/ZONE ==========
    df["is_male"] = (df["gender"] == "Male").astype(int)
    df["is_female"] = (df["gender"] == "Female").astype(int)
    df["is_rural"] = (df["zone"] == "Rural").astype(int)
    df["is_urban"] = (df["zone"].isin(["Urban", "Peri_urban"])).astype(int)
    
    # Gender interactions
    df["male_rural"] = df["is_male"] * df["is_rural"]
    df["female_rural"] = df["is_female"] * df["is_rural"]
    
    # Drop date
    df = df.drop(columns=["deathdate"], errors='ignore')
    
    return df


def add_location_target_encoding(train_df, test_df, target_col, k_folds=5):
    """Add target encoding for location features with smoothing"""
    train_df = train_df.copy()
    test_df = test_df.copy()
    
    global_mean = train_df[target_col].mean()
    
    # Location-based target encoding
    for loc_col in ['location_hash', 'region_quad', 'zone']:
        if loc_col in train_df.columns:
            # Calculate smoothed target encoding
            agg = train_df.groupby(loc_col)[target_col].agg(['mean', 'count'])
            # Smoothing: weighted average of local mean and global mean
            smoothing = 10
            train_df[f'{loc_col}_target_enc'] = (
                agg['count'] * agg['mean'] + smoothing * global_mean
            ) / (agg['count'] + smoothing)
            train_df[f'{loc_col}_target_enc'] = train_df[loc_col].map(
                dict(zip(agg.index, 
                    (agg['count'] * agg['mean'] + smoothing * global_mean) / (agg['count'] + smoothing)))
            )
            test_df[f'{loc_col}_target_enc'] = test_df[loc_col].map(
                dict(zip(agg.index,
                    (agg['count'] * agg['mean'] + smoothing * global_mean) / (agg['count'] + smoothing)))
            ).fillna(global_mean)
    
    return train_df, test_df


def add_kmeans_clusters(train_df, test_df, n_clusters=8):
    """Add KMeans clusters based on coordinates"""
    train_df = train_df.copy()
    test_df = test_df.copy()
    
    coords = ['latitude', 'longitude']
    
    # Fit on combined data for consistent clusters
    all_coords = pd.concat([train_df[coords], test_df[coords]], ignore_index=True)
    kmeans = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=10)
    
    train_df['cluster'] = kmeans.fit_predict(train_df[coords])
    test_df['cluster'] = kmeans.predict(test_df[coords])
    
    # Add cluster center distances
    for i in range(n_clusters):
        center = kmeans.cluster_centers_[i]
        train_df[f'dist_cluster_{i}'] = np.sqrt(
            (train_df['latitude'] - center[0])**2 + 
            (train_df['longitude'] - center[1])**2
        )
        test_df[f'dist_cluster_{i}'] = np.sqrt(
            (test_df['latitude'] - center[0])**2 + 
            (test_df['longitude'] - center[1])**2
        )
    
    return train_df, test_df, kmeans


# Apply feature engineering
print("=" * 60)
print("GEOSPATIAL FEATURE ENGINEERING")
print("=" * 60)

train = deep_feature_engineering(train, is_train=True)
test = deep_feature_engineering(test, is_train=False)

# Add KMeans clusters
train, test, kmeans = add_kmeans_clusters(train, test, n_clusters=N_CLUSTERS)

# Add target encoding (using full train data - for final model)
train, test = add_location_target_encoding(train, test, TARGET)

print(f"Features after engineering: {train.shape[1] - 2}")
print(f"New features created: ~{train.shape[1] - 29}")

## 📊 Feature Selection

In [ ]:
# Identify feature columns
exclude_cols = [ID_COL, TARGET, "location", "location_hash", "lat_bin_fine", "lon_bin_fine"]
feature_cols = [c for c in train.columns if c not in exclude_cols]

X = train[feature_cols].copy()
y = train[TARGET].copy()
X_test = test[feature_cols].copy()

# Categorical and numeric
cat_cols = ['zone', 'gender', 'cluster']
num_cols = [c for c in feature_cols if c not in cat_cols]

print(f"Total features: {len(feature_cols)}")
print(f"Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}")

# Remove highly correlated features
def remove_highly_correlated(X, threshold=0.95):
    corr_matrix = X.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    print(f"Removing {len(to_drop)} highly correlated features")
    return X.drop(columns=to_drop), to_drop

X_filtered, dropped = remove_highly_correlated(X[num_cols], threshold=0.95)
num_cols_filtered = list(X_filtered.columns)
feature_cols_filtered = num_cols_filtered + cat_cols

X = X[feature_cols_filtered]
X_test = X_test[feature_cols_filtered]

print(f"Features after correlation filter: {len(feature_cols_filtered)}")

## 🔧 Preprocessing Pipeline

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_cols),
        ("num", Pipeline([
            ("imputer", KNNImputer(n_neighbors=5)),
            ("scaler", RobustScaler())
        ]), num_cols_filtered)
    ]
)

## 🤖 Optimized Model Definitions

In [ ]:
def get_optimized_models():
    """Return optimized models with tuned parameters"""
    models = {}
    
    # XGBoost - Optimized
    if HAS_XGB:
        models['XGBoost'] = Pipeline([
            ("preprocess", preprocessor),
            ("model", xgb.XGBClassifier(
                n_estimators=400,
                max_depth=6,
                learning_rate=0.02,
                subsample=0.85,
                colsample_bytree=0.85,
                colsample_bylevel=0.85,
                min_child_weight=2,
                gamma=0.05,
                reg_alpha=0.05,
                reg_lambda=0.8,
                scale_pos_weight=1.5,
                random_state=RANDOM_STATE,
                eval_metric='auc',
                tree_method='hist',
                n_jobs=-1
            ))
        ])
        
        models['XGBoost_Deep'] = Pipeline([
            ("preprocess", preprocessor),
            ("model", xgb.XGBClassifier(
                n_estimators=400,
                max_depth=8,
                learning_rate=0.015,
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_weight=1,
                scale_pos_weight=1.8,
                random_state=RANDOM_STATE,
                eval_metric='auc',
                tree_method='hist',
                n_jobs=-1
            ))
        ])
    
    # LightGBM - Optimized
    if HAS_LGBM:
        models['LightGBM'] = Pipeline([
            ("preprocess", preprocessor),
            ("model", lgb.LGBMClassifier(
                n_estimators=400,
                max_depth=7,
                learning_rate=0.02,
                num_leaves=24,
                subsample=0.85,
                colsample_bytree=0.85,
                min_child_samples=15,
                reg_alpha=0.05,
                reg_lambda=0.8,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                verbose=-1,
                n_jobs=-1
            ))
        ])
    
    # CatBoost - Optimized
    if HAS_CAT:
        models['CatBoost'] = Pipeline([
            ("preprocess", preprocessor),
            ("model", cb.CatBoostClassifier(
                iterations=400,
                depth=6,
                learning_rate=0.02,
                l2_leaf_reg=2.0,
                class_weights="balanced",
                random_state=RANDOM_STATE,
                verbose=0,
                thread_count=-1
            ))
        ])
    
    # Random Forest - Optimized
    models['RandomForest'] = Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=400,
            max_depth=12,
            min_samples_split=3,
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
    
    # Extra Trees - Optimized
    models['ExtraTrees'] = Pipeline([
        ("preprocess", preprocessor),
        ("model", ExtraTreesClassifier(
            n_estimators=400,
            max_depth=12,
            min_samples_split=3,
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
    
    # Gradient Boosting - Optimized
    models['GradientBoosting'] = Pipeline([
        ("preprocess", preprocessor),
        ("model", GradientBoostingClassifier(
            n_estimators=250,
            max_depth=5,
            learning_rate=0.03,
            min_samples_split=8,
            min_samples_leaf=4,
            subsample=0.85,
            random_state=RANDOM_STATE
        ))
    ])
    
    # HistGradientBoosting
    models['HistGB'] = Pipeline([
        ("preprocess", preprocessor),
        ("model", HistGradientBoostingClassifier(
            max_iter=350,
            max_depth=7,
            learning_rate=0.03,
            l2_regularization=0.05,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ])
    
    # Calibrated Logistic Regression
    models['LogisticRegression'] = Pipeline([
        ("preprocess", preprocessor),
        ("model", CalibratedClassifierCV(
            LogisticRegression(
                max_iter=2000,
                C=0.3,
                class_weight="balanced",
                solver="lbfgs",
                random_state=RANDOM_STATE
            ),
            method="isotonic",
            cv=5
        ))
    ])
    
    # MLP - Optimized
    models['MLP'] = Pipeline([
        ("preprocess", preprocessor),
        ("model", MLPClassifier(
            hidden_layer_sizes=(100, 50, 25),
            activation='relu',
            solver='adam',
            alpha=0.005,
            batch_size=64,
            learning_rate='adaptive',
            max_iter=500,
            random_state=RANDOM_STATE,
            early_stopping=True,
            validation_fraction=0.1
        ))
    ])
    
    return models

models = get_optimized_models()
print(f"Models configured: {list(models.keys())}")

## 📈 Cross-Validation Evaluation

In [ ]:
print("=" * 60)
print("CROSS-VALIDATION EVALUATION")
print("=" * 60)

cv_results = {}
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

for name, model in models.items():
    print(f"\nEvaluating {name}...")
    
    try:
        cv_proba = cross_val_predict(model, X, y, cv=skf, method="predict_proba", n_jobs=-1)[:, 1]
        cv_pred = (cv_proba >= 0.5).astype(int)
        
        f1 = f1_score(y, cv_pred)
        auc = roc_auc_score(y, cv_proba)
        final_score = 0.60 * f1 + 0.40 * auc
        
        cv_results[name] = {
            'model': model,
            'f1': f1,
            'auc': auc,
            'final_score': final_score,
            'cv_proba': cv_proba
        }
        
        print(f"  F1: {f1:.4f} | AUC: {auc:.4f} | Final: {final_score:.4f}")
    except Exception as e:
        print(f"  Error: {e}")

# Results table
print("\n" + "=" * 60)
print("CV RESULTS")
print("=" * 60)

if cv_results:
    results_df = pd.DataFrame({
        'Model': list(cv_results.keys()),
        'F1': [cv_results[m]['f1'] for m in cv_results],
        'AUC': [cv_results[m]['auc'] for m in cv_results],
        'Final': [cv_results[m]['final_score'] for m in cv_results]
    }).sort_values('Final', ascending=False)
    
    print(results_df.to_string(index=False))
else:
    print("No models evaluated!")

## 🎯 Threshold Optimization

In [ ]:
print("=" * 60)
print("THRESHOLD OPTIMIZATION")
print("=" * 60)

if cv_results:
    best_model_name = max(cv_results, key=lambda x: cv_results[x]['final_score'])
    print(f"Best model: {best_model_name}")
    
    best_proba = cv_results[best_model_name]['cv_proba']
    
    def objective(threshold):
        pred = (best_proba >= threshold).astype(int)
        f1_t = f1_score(y, pred)
        auc_t = roc_auc_score(y, best_proba)
        return -(0.60 * f1_t + 0.40 * auc_t)
    
    result = minimize_scalar(objective, bounds=(0.3, 0.7), method='bounded')
    best_threshold = result.x
    best_score = -result.fun
    
    optimal_pred = (best_proba >= best_threshold).astype(int)
    optimal_f1 = f1_score(y, optimal_pred)
    
    print(f"\nOptimal threshold: {best_threshold:.4f}")
    print(f"F1 at optimal: {optimal_f1:.4f}")
    print(f"Best combined score: {best_score:.4f}")
else:
    best_threshold = 0.5
    print("Using default threshold: 0.5")

## 🏆 Stacking Ensemble

In [ ]:
print("=" * 60)
print("STACKING ENSEMBLE")
print("=" * 60)

if cv_results:
    # Select top 5 models
    top_models = sorted(cv_results.keys(), key=lambda x: cv_results[x]['final_score'], reverse=True)[:5]
    print(f"Top models: {top_models}")
    
    base_models = [(name, cv_results[name]['model']) for name in top_models]
    
    # Better meta-learner
    meta_learner = LogisticRegression(
        max_iter=2000,
        C=0.5,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
    
    stacking_clf = StackingClassifier(
        estimators=base_models,
        final_estimator=meta_learner,
        cv=5,
        stack_method="predict_proba",
        n_jobs=-1
    )
    
    print("\nEvaluating Stacking...")
    stacking_proba = cross_val_predict(stacking_clf, X, y, cv=skf, method="predict_proba", n_jobs=-1)[:, 1]
    stacking_pred = (stacking_proba >= best_threshold).astype(int)
    
    stacking_f1 = f1_score(y, stacking_pred)
    stacking_auc = roc_auc_score(y, stacking_proba)
    stacking_final = 0.60 * stacking_f1 + 0.40 * stacking_auc
    
    print(f"  F1: {stacking_f1:.4f} | AUC: {stacking_auc:.4f} | Final: {stacking_final:.4f}")
    
    cv_results['Stacking'] = {
        'model': stacking_clf,
        'f1': stacking_f1,
        'auc': stacking_auc,
        'final_score': stacking_final,
        'cv_proba': stacking_proba
    }
    
    # Update results
    results_df = pd.DataFrame({
        'Model': list(cv_results.keys()),
        'F1': [cv_results[m]['f1'] for m in cv_results],
        'AUC': [cv_results[m]['auc'] for m in cv_results],
        'Final': [cv_results[m]['final_score'] for m in cv_results]
    }).sort_values('Final', ascending=False)
    
    print("\nUpdated Results:")
    print(results_df.to_string(index=False))

## 🎲 Ensemble Predictions

In [ ]:
print("=" * 60)
print("ENSEMBLE PREDICTIONS")
print("=" * 60)

if cv_results:
    # Model weights
    total_score = sum(cv_results[m]['final_score'] for m in cv_results)
    model_weights = {m: cv_results[m]['final_score'] / total_score for m in cv_results}
    
    print("Model weights:")
    for m, w in sorted(model_weights.items(), key=lambda x: -x[1])[:5]:
        print(f"  {m:20s}: {w:.4f}")
    
    # Weighted ensemble
    ensemble_proba = np.zeros(len(X_test))
    
    for name, model in models.items():
        if name in cv_results:
            print(f"Training {name}...")
            model.fit(X, y)
            test_proba = model.predict_proba(X_test)[:, 1]
            ensemble_proba += model_weights[name] * test_proba
    
    # Simple average
    all_probas = []
    for name, model in models.items():
        if name in cv_results:
            model.fit(X, y)
            all_probas.append(model.predict_proba(X_test)[:, 1])
    simple_avg = np.mean(all_probas, axis=0)
    
    # Final blend
    if 'Stacking' in cv_results:
        print("Training Stacking...")
        stacking_clf.fit(X, y)
        stacking_proba_test = stacking_clf.predict_proba(X_test)[:, 1]
        final_proba = 0.5 * ensemble_proba + 0.3 * simple_avg + 0.2 * stacking_proba_test
    else:
        final_proba = 0.7 * ensemble_proba + 0.3 * simple_avg
    
    final_pred = (final_proba >= best_threshold).astype(int)
    
    print(f"\nEnsemble range: [{final_proba.min():.3f}, {final_proba.max():.3f}]")
    print(f"Predicted positives: {final_pred.sum()} / {len(final_pred)} ({100*final_pred.mean():.1f}%)")

## 📊 Feature Importance

In [ ]:
print("=" * 60)
print("FEATURE IMPORTANCE")
print("=" * 60)

if cv_results:
    best_name = max(cv_results, key=lambda x: cv_results[x]['final_score'])
    if best_name == 'Stacking':
        best_name = list(cv_results.keys())[0]
    
    best_model = models[best_name]
    best_model.fit(X, y)
    
    try:
        trained = best_model.named_steps['model']
        if hasattr(trained, 'feature_importances_'):
            ohe = best_model.named_steps['preprocess'].named_transformers_['cat'].named_steps['onehot']
            cat_names = ohe.get_feature_names_out(cat_cols).tolist()
            feature_names = cat_names + num_cols_filtered
            
            imp_df = pd.DataFrame({
                'feature': feature_names,
                'importance': trained.feature_importances_
            }).sort_values('importance', ascending=False)
            
            plt.figure(figsize=(10, 10))
            top = imp_df.head(25)
            plt.barh(range(len(top)), top['importance'].values)
            plt.yticks(range(len(top)), top['feature'].values)
            plt.xlabel('Importance')
            plt.title(f'Top 25 Features ({best_name})')
            plt.gca().invert_yaxis()
            plt.tight_layout()
            plt.show()
            
            print("\nTop 15 features:")
            for _, row in imp_df.head(15).iterrows():
                print(f"  {row['feature']:30s}: {row['importance']:.4f}")
    except Exception as e:
        print(f"Could not extract: {e}")

## 📝 Submission

In [ ]:
print("=" * 60)
print("CREATING SUBMISSION")
print("=" * 60)

submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    "TargetF1": final_pred,
    "TargetRAUC": final_proba
})

submission.to_csv("geospatial_optimized_submission.csv", index=False)
print(f"Saved: geospatial_optimized_submission.csv")
print(f"Shape: {submission.shape}")
print(f"\nTargetF1:\n{submission['TargetF1'].value_counts()}")
print(f"\nTargetRAUC:\n{submission['TargetRAUC'].describe()}")

## 📋 Summary

In [ ]:
print("=" * 60)
print("🏆 GEOSPATIAL OPTIMIZED MODEL")
print("=" * 60)

if cv_results:
    best = max(cv_results, key=lambda x: cv_results[x]['final_score'])
    
    print("\nResults:")
    for name in results_df['Model']:
        r = cv_results[name]
        print(f"{name:20s} | F1: {r['f1']:.4f} | AUC: {r['auc']:.4f} | Score: {r['final_score']:.4f}")
    
    print(f"\nBest: {best} (Score: {cv_results[best]['final_score']:.4f})")
    print(f"Threshold: {best_threshold:.4f}")
    print(f"Submission: geospatial_optimized_submission.csv")
    
    print("\n" + "=" * 60)
    print("KEY IMPROVEMENTS")
    print("=" * 60)
    print("""
1. GEOSPATIAL FEATURES:
   - KMeans clustering (8 clusters) on coordinates
   - Distance from Uganda center
   - Distance from each cluster center
   - Regional encoding (N/S, E/W quadrants)
   - Fine-grained lat/lon binning
   - Target encoding for location features

2. OPTIMIZED MODELS:
   - XGBoost (2 variants) with tuned params
   - LightGBM with optimized depth/leaves
   - CatBoost with tuned regularization
   - RandomForest/ExtraTrees (400 trees)
   - HistGradientBoosting
   - Calibrated LogisticRegression
   - MLP Neural Network

3. ENSEMBLE:
   - Stacking with top 5 models
   - Weighted blend: 50% + 30% + 20%
   - Threshold optimization
    """)